# Cricket Sentiment Analysis

This notebook performs sentiment analysis on cricket-related text data (tweets, comments, articles).

## Objectives:
- Collect social media data about cricket matches/players
- Perform sentiment analysis
- Visualize fan sentiment trends
- Correlate sentiment with match outcomes

**Note:** This is an optional feature for advanced analytics.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print("\nNote: Sentiment analysis requires additional setup:")
print("- Twitter API credentials (for tweet collection)")
print("- NLP libraries (transformers, nltk, textblob)")
print("- Pre-trained sentiment models")

## 1. Data Collection (Placeholder)

This section would typically collect data from:
- Twitter API
- Reddit API
- News articles
- Fan forums

In [ ]:
# Placeholder for data collection
# In production, you would use:
# - tweepy for Twitter
# - praw for Reddit
# - Beautiful Soup for web scraping

sample_data = pd.DataFrame({
    'text': [
        'Amazing century by the captain!',
        'Terrible bowling performance today',
        'What a thrilling match!',
        'Disappointed with the result',
        'Best IPL season ever!'
    ],
    'timestamp': pd.date_range('2024-01-01', periods=5, freq='D'),
    'match_id': ['M1', 'M1', 'M2', 'M2', 'M3']
})

print(f"Sample data shape: {sample_data.shape}")
display(sample_data)

## 2. Text Preprocessing

In [ ]:
import re

def clean_text(text):
    """
    Clean and preprocess text data.
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove special characters and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

sample_data['cleaned_text'] = sample_data['text'].apply(clean_text)
display(sample_data[['text', 'cleaned_text']])

## 3. Sentiment Analysis

Multiple approaches can be used:
1. Rule-based (VADER, TextBlob)
2. Machine Learning (trained classifier)
3. Deep Learning (BERT, RoBERTa)

In [ ]:
# Example using TextBlob (simple sentiment analyzer)
try:
    from textblob import TextBlob
    
    def get_sentiment(text):
        """
        Get sentiment polarity using TextBlob.
        Returns: polarity (-1 to 1), sentiment label
        """
        blob = TextBlob(text)
        polarity = blob.sentiment.polarity
        
        if polarity > 0.1:
            sentiment = 'positive'
        elif polarity < -0.1:
            sentiment = 'negative'
        else:
            sentiment = 'neutral'
        
        return polarity, sentiment
    
    # Apply sentiment analysis
    sample_data[['polarity', 'sentiment']] = sample_data['cleaned_text'].apply(
        lambda x: pd.Series(get_sentiment(x))
    )
    
    display(sample_data[['text', 'polarity', 'sentiment']])
    
except ImportError:
    print("TextBlob not installed. Install with: pip install textblob")
    print("For now, using placeholder sentiment scores")
    
    # Placeholder sentiment
    sample_data['polarity'] = np.random.uniform(-1, 1, len(sample_data))
    sample_data['sentiment'] = sample_data['polarity'].apply(
        lambda x: 'positive' if x > 0.1 else ('negative' if x < -0.1 else 'neutral')
    )

## 4. Sentiment Visualization

In [ ]:
# Sentiment distribution
if 'sentiment' in sample_data.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart
    sentiment_counts = sample_data['sentiment'].value_counts()
    axes[0].pie(sentiment_counts, labels=sentiment_counts.index, autopct='%1.1f%%',
               colors=['#90EE90', '#FFB6C1', '#ADD8E6'])
    axes[0].set_title('Sentiment Distribution')
    
    # Polarity distribution
    axes[1].hist(sample_data['polarity'], bins=20, color='skyblue', edgecolor='black')
    axes[1].set_title('Polarity Score Distribution')
    axes[1].set_xlabel('Polarity')
    axes[1].set_ylabel('Frequency')
    axes[1].axvline(x=0, color='red', linestyle='--', label='Neutral')
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

## 5. Sentiment Trends Over Time

In [ ]:
# Time-based sentiment analysis
if 'timestamp' in sample_data.columns and 'polarity' in sample_data.columns:
    # Group by date and calculate average sentiment
    daily_sentiment = sample_data.groupby(sample_data['timestamp'].dt.date)['polarity'].mean()
    
    plt.figure(figsize=(12, 6))
    daily_sentiment.plot(kind='line', marker='o', color='purple')
    plt.axhline(y=0, color='red', linestyle='--', alpha=0.5, label='Neutral')
    plt.title('Average Sentiment Over Time')
    plt.xlabel('Date')
    plt.ylabel('Average Polarity')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 6. Sentiment by Match/Team

In [ ]:
# Sentiment by match
if 'match_id' in sample_data.columns and 'sentiment' in sample_data.columns:
    match_sentiment = pd.crosstab(sample_data['match_id'], sample_data['sentiment'])
    
    match_sentiment.plot(kind='bar', stacked=True, figsize=(10, 6),
                        color=['#90EE90', '#FFB6C1', '#ADD8E6'])
    plt.title('Sentiment Distribution by Match')
    plt.xlabel('Match ID')
    plt.ylabel('Count')
    plt.legend(title='Sentiment')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 7. Word Cloud Analysis

In [ ]:
try:
    from wordcloud import WordCloud
    
    # Separate positive and negative texts
    positive_text = ' '.join(sample_data[sample_data['sentiment'] == 'positive']['cleaned_text'])
    negative_text = ' '.join(sample_data[sample_data['sentiment'] == 'negative']['cleaned_text'])
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Positive word cloud
    if positive_text:
        wc_positive = WordCloud(width=800, height=400, background_color='white',
                               colormap='Greens').generate(positive_text)
        axes[0].imshow(wc_positive, interpolation='bilinear')
        axes[0].set_title('Positive Sentiment Words', fontsize=16)
        axes[0].axis('off')
    
    # Negative word cloud
    if negative_text:
        wc_negative = WordCloud(width=800, height=400, background_color='white',
                               colormap='Reds').generate(negative_text)
        axes[1].imshow(wc_negative, interpolation='bilinear')
        axes[1].set_title('Negative Sentiment Words', fontsize=16)
        axes[1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("WordCloud not installed. Install with: pip install wordcloud")

## 8. Correlation with Match Outcomes

In [ ]:
# This would require match outcome data
# Example: Does pre-match sentiment correlate with actual results?

print("\nInsight: Sentiment Correlation with Match Outcomes")
print("="*60)
print("To analyze:")
print("1. Pre-match sentiment vs actual result")
print("2. Sentiment shift during match vs comeback probability")
print("3. Post-match sentiment intensity vs margin of victory")
print("\nRequires: Integration with match results data from CricPy API")

## 9. Advanced Sentiment Analysis (Using Transformers)

In [ ]:
# Example using pre-trained BERT model for sentiment
try:
    from transformers import pipeline
    
    # Load pre-trained sentiment analysis pipeline
    sentiment_pipeline = pipeline('sentiment-analysis')
    
    # Analyze sample texts
    texts = sample_data['text'].tolist()[:3]  # Limit for demo
    results = sentiment_pipeline(texts)
    
    print("\nBERT-based Sentiment Analysis:")
    for text, result in zip(texts, results):
        print(f"Text: {text}")
        print(f"Sentiment: {result['label']}, Score: {result['score']:.4f}\n")
    
except ImportError:
    print("Transformers library not installed.")
    print("Install with: pip install transformers torch")
except Exception as e:
    print(f"Error: {e}")
    print("Note: This requires significant resources and internet connection")

## 10. Export Results

In [ ]:
# Save sentiment analysis results
output_path = Path.cwd().parent / 'data' / 'processed' / 'sentiment_results.csv'

if not sample_data.empty:
    sample_data.to_csv(output_path, index=False)
    print(f"Sentiment results saved to: {output_path}")
    print(f"\nSummary Statistics:")
    print(f"Total texts analyzed: {len(sample_data)}")
    if 'sentiment' in sample_data.columns:
        print(f"\nSentiment breakdown:")
        print(sample_data['sentiment'].value_counts())
        print(f"\nAverage polarity: {sample_data['polarity'].mean():.3f}")

## Next Steps

1. **Real-time Monitoring** - Set up live sentiment tracking during matches
2. **API Integration** - Connect to Twitter/Reddit APIs for data collection
3. **Advanced Models** - Fine-tune BERT on cricket-specific data
4. **Dashboard Integration** - Add sentiment widgets to Streamlit dashboard
5. **Predictive Analysis** - Use sentiment as a feature in match prediction models